# Functions, Return Values & Exceptions

## What you'll practice

Functions that return useful things (tuples, `None`, collections of errors), defaults, and exceptions used
the Python way: specific exception types, custom hierarchies, `raise ... from ...`, and EAFP
("easier to ask forgiveness than permission"). Type hints start appearing here — lightly.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/fundamentals/test_01_04_functions_and_exceptions.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/01_python_fundamentals/04_functions_and_exceptions.ipynb`. Don't open it until you've tried.

## Exercise 1 — 🟢 Easy · Safe integer conversion

**Implement:** `safe_int(value, default=None)`

Convert `value` to `int`. Accept ints and strings of digits with optional sign and surrounding whitespace.
Return `default` for `None`, blank strings, floats-as-strings (`"3.5"`, `"3.0"`), garbage, and `bool`s.

**Input:** `value, default=None`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
safe_int(" 42 ")   # 42
safe_int("-7")     # -7
safe_int("3.0", 0) # 0
safe_int(None, -1) # -1
```

**Check:** `pytest tests/fundamentals/test_01_04_functions_and_exceptions.py -k "test_safe_int_ok or test_safe_int_default"`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Collect validation errors

**Implement:** `validate_user(record: dict) -> list[str]`

Return a list of human-readable problems (empty list = valid). Check, in this order:
`name` present and non-blank → `"name is required"`; `email` present and contains exactly one `@` → `"email is invalid"`;
`age` is an int (not bool) between 18 and 120 inclusive → `"age must be an integer between 18 and 120"`.
Report **all** problems, not just the first.

**Input:** `record: dict`

**Expected output:** `list[str]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
validate_user({"name": "Kalyan", "email": "k@x.io", "age": 23})   # []
validate_user({"email": "nope"})   # ['name is required', 'email is invalid', 'age must be an integer between 18 and 120']
```

**Check:** `pytest tests/fundamentals/test_01_04_functions_and_exceptions.py -k "test_validate_user_ok or test_validate_user_all_errors or test_validate_user_age or test_validate_user_blank_name"`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Parse records, separating good and bad

**Implement:** `parse_records(lines: list[str]) -> tuple[list[dict], list[tuple[int, str]]]`

Each line is `id,name,amount`. Return `(good, bad)`: `good` is a list of `{'id': int, 'name': str, 'amount': float}`,
`bad` is a list of `(line_number, error_message)` with 1-based line numbers. A line is bad if it doesn't have
exactly 3 fields, if `id` isn't an int, or if `amount` isn't a number. Blank lines are skipped silently.
Return both lists as a tuple — this is Python's idiomatic "multiple return values".

**Input:** `lines: list[str]`

**Expected output:** `tuple[list[dict], list[tuple[int, str]]]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
good, bad = parse_records(["1,Laptop,70000", "x,Mouse,10", "", "3,Desk"])
# good == [{'id': 1, 'name': 'Laptop', 'amount': 70000.0}]; [n for n, _ in bad] == [2, 4]
```

**Check:** `pytest tests/fundamentals/test_01_04_functions_and_exceptions.py -k "test_parse_records or test_parse_records_empty"`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Custom exceptions with context and chaining

**Implement:** `class RecordValidationError(ValueError)`, `validate_order(order: dict) -> dict`

Create `RecordValidationError(ValueError)` whose constructor takes `(field, message, value=None)`, stores them as
attributes, and whose `str()` is `"field 'quantity': must be positive (got -1)"` (omit ` (got ...)` when value is `None`).

`validate_order(order)` returns a **new** normalized dict `{'order_id': str, 'quantity': int, 'unit_price': float}` or raises
`RecordValidationError` for the first problem:
- `order_id` missing/blank → field `order_id`, message `"is required"`
- `quantity` not convertible to int → message `"must be an integer"`, and the exception must be **chained** from the original
  `ValueError` (`raise ... from exc`), so `err.__cause__` is set
- `quantity <= 0` → `"must be positive"`
- `unit_price` not convertible to float or negative → `"must be a non-negative number"`

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
validate_order({"order_id": " O1 ", "quantity": "2", "unit_price": "10.5"})
# {'order_id': 'O1', 'quantity': 2, 'unit_price': 10.5}
```

**Check:** `pytest tests/fundamentals/test_01_04_functions_and_exceptions.py -k "test_record_validation_error_str or test_validate_order_ok or test_validate_order_errors or test_validate_order_chains_cause"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Batch processing with a failure budget

**Implement:** `class BatchFailed(Exception)`, `process_batch(records, transform, max_failure_ratio=0.1) -> dict`

Apply `transform(record)` to every record. Collect outputs and failures; never let one bad record stop the batch.
Return `{'output': [...], 'processed': int, 'failed': int, 'errors': [{'index': i, 'error_type': 'ValueError', 'message': '...'}]}`.

- Catch `Exception` only — `KeyboardInterrupt`/`SystemExit` must propagate.
- If `failed / total > max_failure_ratio`, raise `BatchFailed` with the summary dict available as `err.summary`.
- An empty batch returns a summary with zeros (no division by zero).

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
process_batch(["1", "x", "3"], int, max_failure_ratio=0.5)
# {'output': [1, 3], 'processed': 2, 'failed': 1, 'errors': [{'index': 1, 'error_type': 'ValueError', 'message': "invalid literal..."}]}
```

**Check:** `pytest tests/fundamentals/test_01_04_functions_and_exceptions.py -k "test_process_batch_ok or test_process_batch_empty or test_process_batch_raises_with_summary or test_process_batch_boundary_ratio or test_process_batch_keyboard_interrupt_propagates"`

**YOUR CODE ↓**